In [2]:
import pandas as pd, numpy as np, os
import xlsxwriter

df = pd.read_csv("data/clean_orders.csv")
th = pd.read_csv("data/theme_summary.csv")          # from the review-themes notebook
ex = pd.read_csv("data/theme_examples_en.csv")      # translated examples
os.makedirs("excel", exist_ok=True)

N      = len(df)
n_late = int(df.is_late.sum())
n_low  = int(df.low_review.sum())
p_late = df[df.is_late == 1].low_review.mean()
p_on   = df[df.is_late == 0].low_review.mean()
excess_share = (p_late - p_on) * n_late / n_low      # your "28%" number
print(N, n_late, n_low, round(p_late, 3), round(p_on, 3), round(excess_share, 3))

95560 6378 12228 0.624 0.092 0.277


In [3]:
df["delay_bucket"] = pd.cut(df.days_late, [-np.inf, 0, 3, 7, np.inf],
                            labels=["On time", "1-3 days late", "4-7 days late", "8+ days late"])
delay = (df.groupby("delay_bucket", observed=True)
           .agg(orders=("order_id", "count"), bad=("low_review", "sum"), avg=("review_score", "mean"))
           .reset_index())

state = (df.groupby("customer_state")
           .agg(orders=("order_id", "count"), late=("is_late", "sum"),
                bad=("low_review", "sum"), avg=("review_score", "mean"))
           .query("orders >= 300")
           .assign(r=lambda d: d.late / d.orders)
           .sort_values("r", ascending=False).reset_index())

cat = (df.groupby("category")
         .agg(orders=("order_id", "count"), bad=("low_review", "sum"), late=("is_late", "sum"))
         .query("orders >= 300")
         .assign(r=lambda d: d.bad / d.orders)
         .sort_values("r", ascending=False).head(12).reset_index())

sel = (df.groupby(["seller_id", "seller_state"])
         .agg(orders=("order_id", "count"), late=("is_late", "mean"),
              bad=("low_review", "mean"), handoff=("seller_handoff_days", "mean"))
         .query("orders >= 50")
         .sort_values("bad", ascending=False).head(10).reset_index())

s = df.groupby("seller_id").is_late.sum().sort_values(ascending=False)
top10_share = s.head(int(len(s) * 0.10)).sum() / s.sum()   # share of late orders from worst 10% of sellers
worst_states = ", ".join(state.customer_state.head(3))

In [4]:
sp_rate = state.loc[state.customer_state == "SP", "r"].iloc[0]
state["extra_late"] = (state.r - sp_rate).clip(lower=0) * state.orders

extra = (state.sort_values("extra_late", ascending=False)
              [["customer_state", "orders", "late", "r", "extra_late"]]
              .round(3).head(8))
print(extra.to_string(index=False))

rj_extra = int(state.loc[state.customer_state == "RJ", "extra_late"].iloc[0])
rj_share = rj_extra / n_late
print(f"\nSP benchmark late rate: {sp_rate:.1%}")
print(f"RJ extra late orders: {rj_extra:,}  ({rj_share:.0%} of all late orders)")
print(f"Bad reviews avoided if RJ matched SP: about {rj_extra * (p_late - p_on):.0f} "
      f"({rj_extra * (p_late - p_on) / n_low:.1%} of all bad reviews)")

customer_state  orders  late     r  extra_late
            RJ   12172  1456 0.120     915.743
            BA    3226   384 0.119     240.813
            SC    3510   284 0.081     128.208
            CE    1267   174 0.137     117.764
            ES    1966   205 0.104     117.739
            MA     709   122 0.172      90.531
            RS    5309   321 0.060      85.359
            PE    1573   149 0.095      79.182

SP benchmark late rate: 4.4%
RJ extra late orders: 915  (14% of all late orders)
Bad reviews avoided if RJ matched SP: about 487 (4.0% of all bad reviews)


In [4]:
wb = xlsxwriter.Workbook("excel/Olist_CX_Report.xlsx")

# ---------- formats ----------
NAVY = "#1F3864"
title  = wb.add_format({"bold": True, "font_size": 18, "font_color": NAVY})
note   = wb.add_format({"italic": True, "font_color": "#595959", "text_wrap": True, "valign": "top"})
body   = wb.add_format({"text_wrap": True, "valign": "top", "font_size": 11})
bold   = wb.add_format({"bold": True, "font_size": 12, "font_color": NAVY})
hdr    = wb.add_format({"bold": True, "bg_color": NAVY, "font_color": "white", "border": 1,
                        "text_wrap": True, "align": "center", "valign": "vcenter"})
cell   = wb.add_format({"border": 1, "text_wrap": True, "valign": "top"})
num    = wb.add_format({"border": 1, "num_format": "#,##0"})
pct    = wb.add_format({"border": 1, "num_format": "0.0%"})
dec    = wb.add_format({"border": 1, "num_format": "0.00"})
card_l = wb.add_format({"align": "center", "valign": "vcenter", "text_wrap": True,
                        "font_color": "#595959", "border": 1, "bg_color": "#F2F2F2"})
card_n = wb.add_format({"align": "center", "valign": "vcenter", "bold": True, "font_size": 22,
                        "border": 1, "bg_color": "#EAF1FB", "font_color": NAVY, "num_format": "#,##0"})
card_p = wb.add_format({"align": "center", "valign": "vcenter", "bold": True, "font_size": 22,
                        "border": 1, "bg_color": "#EAF1FB", "font_color": NAVY, "num_format": "0.0%"})
card_x = wb.add_format({"align": "center", "valign": "vcenter", "bold": True, "font_size": 22,
                        "border": 1, "bg_color": "#EAF1FB", "font_color": NAVY, "num_format": '0.0"x"'})
card_d = wb.add_format({"align": "center", "valign": "vcenter", "bold": True, "font_size": 22,
                        "border": 1, "bg_color": "#EAF1FB", "font_color": NAVY, "num_format": "0.00"})
SCALE = {"type": "3_color_scale", "min_color": "#63BE7B", "mid_color": "#FFEB84", "max_color": "#F8696B"}

def setup(ws, tab):
    ws.hide_gridlines(2); ws.set_tab_color(tab)

# ================= 1. READ ME FIRST =================
ws = wb.add_worksheet("Read Me First"); setup(ws, NAVY)
ws.set_column("A:A", 3); ws.set_column("B:B", 28); ws.set_column("C:C", 80)
ws.write("B1", "Olist Customer Experience Report", title)
ws.write("B2", "What is driving bad reviews, and what should we do?", note)
ws.write("B4", "THE ANSWER IN 3 LINES", bold)
ws.merge_range("B5:C5", f"1. Late orders get a 1-2 star review {p_late/p_on:.1f}x as often as on-time orders "
                        f"({p_late:.0%} vs {p_on:.0%}). The longer the delay, the worse the rating.", body)
ws.merge_range("B6:C6", f"2. But late delivery explains only about {excess_share:.0%} of bad reviews. "
                        f"Most come from orders that arrived on time: wrong or missing items, quality, returns.", body)
ws.merge_range("B7:C7", f"3. The delay problem is concentrated: the worst states are {worst_states}, and the "
                        f"worst 10% of sellers cause {top10_share:.0%} of late orders.", body)
for r in (4, 5, 6): ws.set_row(r, 34)

ws.write("B9", "HOW TO USE THIS FILE", bold)
guide = [("Key Numbers", "The big picture on one screen"),
         ("Delay Impact", "How reviews get worse as delays get longer"),
         ("By State", "Where deliveries are late (red = worse). Sorted worst first"),
         ("By Category", "Which product types get the most bad reviews, and whether delivery is the cause"),
         ("Sellers", "The 10 sellers with the highest bad-review rate"),
         ("Complaint Themes", "What customers write in their reviews, with English examples"),
         ("Early Warning and Actions", "Which orders are risky, and the 3 recommended actions")]
for i, (a, b) in enumerate(guide):
    ws.write(9 + i, 1, a, cell); ws.write(9 + i, 2, b, cell)

ws.write("B18", "WORDS USED", bold)
defs = [("Bad review", "A 1 or 2 star review"),
        ("Late order", "Delivered after the date promised to the customer"),
        ("Data", f"{N:,} delivered orders with a review, Jan 2017 to Aug 2018 (public Olist dataset)"),
        ("Important", "These results show strong links, not proof of cause. Other factors can play a part.")]
for i, (a, b) in enumerate(defs):
    ws.write(18 + i, 1, a, cell); ws.write(18 + i, 2, b, cell)

# ================= 2. KEY NUMBERS =================
ws = wb.add_worksheet("Key Numbers"); setup(ws, "#2E75B6")
ws.set_column("A:A", 3); ws.set_column("B:E", 24)
ws.write("B1", "Key numbers", title)
cards = [("Orders analysed", N, card_n), ("Average review score (out of 5)", df.review_score.mean(), card_d),
         ("Orders with a bad review", df.low_review.mean(), card_p), ("Orders delivered late", df.is_late.mean(), card_p),
         ("Bad-review rate when ON TIME", p_on, card_p), ("Bad-review rate when LATE", p_late, card_p),
         ("Late orders get bad reviews", p_late / p_on, card_x),
         ("Bad reviews linked to late delivery", excess_share, card_p)]
for i, (lab, val, f) in enumerate(cards):
    row = 3 if i < 4 else 7
    col = 1 + (i % 4)
    ws.write(row, col, lab, card_l)
    ws.write(row + 1, col, float(val), f)
for r in (3, 7): ws.set_row(r, 32)
for r in (4, 8): ws.set_row(r, 48)
ws.merge_range("B11:E12", "How to read this: about 1 in 10 on-time orders gets a bad review, versus about 6 in 10 late "
               "orders. Late orders are rare, so fixing them helps but will not remove most bad reviews.", note)

# ================= 3. DELAY IMPACT =================
ws = wb.add_worksheet("Delay Impact"); setup(ws, "#C0504D")
ws.set_column("A:A", 20); ws.set_column("B:E", 16)
ws.write("A1", "Does lateness hurt reviews?", title)
ws.write("A2", "Even 1-3 days late costs about a full star. Rates are calculated with Excel formulas.", note)
for c, n in enumerate(["Delivery", "Orders", "Bad reviews", "Bad-review rate", "Avg score (1-5)"]):
    ws.write(3, c, n, hdr)
for i, r in delay.iterrows():
    row = 4 + i; e = row + 1
    ws.write(row, 0, str(r.delay_bucket), cell)
    ws.write(row, 1, int(r.orders), num); ws.write(row, 2, int(r.bad), num)
    ws.write_formula(row, 3, f"=C{e}/B{e}", pct, float(r.bad / r.orders))
    ws.write(row, 4, float(r.avg), dec)
last = 4 + len(delay) - 1
ws.conditional_format(4, 3, last, 3, SCALE)
ch = wb.add_chart({"type": "column"})
ch.add_series({"name": "Bad-review rate", "categories": ["Delay Impact", 4, 0, last, 0],
               "values": ["Delay Impact", 4, 3, last, 3], "fill": {"color": "#C0504D"},
               "data_labels": {"value": True, "num_format": "0%"}})
ch.set_title({"name": "Share of 1-2 star reviews, by delay"}); ch.set_legend({"none": True})
ch.set_y_axis({"num_format": "0%", "min": 0, "max": 1}); ch.set_size({"width": 560, "height": 320})
ws.insert_chart("A11", ch)

# ================= 4. BY STATE =================
ws = wb.add_worksheet("By State"); setup(ws, "#E36C09")
ws.set_column("A:A", 10); ws.set_column("B:G", 15)
ws.write("A1", "Where is the delivery problem?", title)
ws.write("A2", "States with at least 300 orders, worst late rate first. Red = worse, green = better.", note)
for c, n in enumerate(["State", "Orders", "Late orders", "Late rate", "Bad reviews", "Bad-review rate", "Avg score"]):
    ws.write(3, c, n, hdr)
for i, r in state.iterrows():
    row = 4 + i; e = row + 1
    ws.write(row, 0, r.customer_state, cell)
    ws.write(row, 1, int(r.orders), num); ws.write(row, 2, int(r.late), num)
    ws.write_formula(row, 3, f"=C{e}/B{e}", pct, float(r.late / r.orders))
    ws.write(row, 4, int(r.bad), num)
    ws.write_formula(row, 5, f"=E{e}/B{e}", pct, float(r.bad / r.orders))
    ws.write(row, 6, float(r.avg), dec)
last = 4 + len(state) - 1
ws.conditional_format(4, 3, last, 3, SCALE); ws.conditional_format(4, 5, last, 5, SCALE)
ws.freeze_panes(4, 1)
ch = wb.add_chart({"type": "bar"})
ch.add_series({"name": "Late rate", "categories": ["By State", 4, 0, last, 0],
               "values": ["By State", 4, 3, last, 3], "fill": {"color": "#E36C09"}})
ch.set_title({"name": "Late delivery rate by state"}); ch.set_legend({"none": True})
ch.set_y_axis({"reverse": True}); ch.set_x_axis({"num_format": "0%"}); ch.set_size({"width": 520, "height": 480})
ws.insert_chart("I4", ch)

# ================= 5. BY CATEGORY =================
ws = wb.add_worksheet("By Category"); setup(ws, "#7F6000")
ws.set_column("A:A", 28); ws.set_column("B:F", 14); ws.set_column("G:G", 34); ws.set_column("I:I", 18)
ws.write("A1", "Is it the products?", title)
ws.write("A2", "The 12 categories with the highest bad-review rate (min 300 orders). The last column is a formula that "
               "compares each category's late rate with the overall late rate.", note)
ws.write("I3", "Overall late rate", card_l); ws.write("I4", float(df.is_late.mean()), pct)
for c, n in enumerate(["Category", "Orders", "Bad reviews", "Bad-review rate", "Late orders", "Late rate", "Diagnosis"]):
    ws.write(5, c, n, hdr)
for i, r in cat.iterrows():
    row = 6 + i; e = row + 1
    ws.write(row, 0, str(r.category), cell)
    ws.write(row, 1, int(r.orders), num); ws.write(row, 2, int(r.bad), num)
    ws.write_formula(row, 3, f"=C{e}/B{e}", pct, float(r.bad / r.orders))
    ws.write(row, 4, int(r.late), num)
    ws.write_formula(row, 5, f"=E{e}/B{e}", pct, float(r.late / r.orders))
    ws.write_formula(row, 6, f'=IF(F{e}>$I$4,"Late deliveries add to the problem","Not mainly delivery: check product and seller")',
                     cell)
last = 6 + len(cat) - 1
ws.conditional_format(6, 3, last, 3, SCALE); ws.freeze_panes(6, 1)

# ================= 6. SELLERS =================
ws = wb.add_worksheet("Sellers"); setup(ws, "#548235")
ws.set_column("A:A", 14); ws.set_column("B:F", 15)
ws.write("A1", "Are a few sellers responsible?", title)
ws.merge_range("A2:F2", f"The worst 10% of sellers cause {top10_share:.0%} of all late orders. Below: the 10 sellers with the "
               f"highest bad-review rate (at least 50 orders). Seller IDs are anonymised codes.", note)
ws.set_row(1, 32)
for c, n in enumerate(["Seller (ID start)", "State", "Orders", "Late rate", "Bad-review rate", "Avg days to hand to carrier"]):
    ws.write(3, c, n, hdr)
for i, r in sel.iterrows():
    row = 4 + i
    ws.write(row, 0, r.seller_id[:8], cell); ws.write(row, 1, str(r.seller_state), cell)
    ws.write(row, 2, int(r.orders), num); ws.write(row, 3, float(r.late), pct)
    ws.write(row, 4, float(r.bad), pct)
    ws.write(row, 5, float(r.handoff) if pd.notna(r.handoff) else "", dec)
last = 4 + len(sel) - 1
ws.conditional_format(4, 4, last, 4, SCALE)

# ================= 7. COMPLAINT THEMES =================
ws = wb.add_worksheet("Complaint Themes"); setup(ws, "#7030A0")
ws.set_column("A:A", 30); ws.set_column("B:D", 16); ws.set_column("F:F", 80)
ws.write("A1", "What are customers saying?", title)
ws.merge_range("A2:D2", "Share of 1-2 star comments that mention each theme (a comment can mention several). "
               "Based on keyword rules, so treat the numbers as approximate.", note)
ws.set_row(1, 45)
t = th.copy(); t["theme"] = t["theme"].replace({"no_theme": "No theme found"})
t = t.fillna(0)
for c, n in enumerate(["Theme", "All complaints", "On-time orders", "Late orders"]):
    ws.write(3, c, n, hdr)
for i, r in t.iterrows():
    row = 4 + i
    ws.write(row, 0, r["theme"], cell); ws.write(row, 1, float(r["pct_of_complaints"]) / 100, pct)
    ws.write(row, 2, float(r["On-time orders"]) / 100, pct); ws.write(row, 3, float(r["Late orders"]) / 100, pct)
last = 4 + len(t) - 1
ws.conditional_format(4, 1, last, 3, {"type": "data_bar", "bar_color": "#B4A7D6"})
ch = wb.add_chart({"type": "bar"})
for col, name, color in [(2, "On-time orders", "#2E75B6"), (3, "Late orders", "#C0504D")]:
    ch.add_series({"name": name, "categories": ["Complaint Themes", 4, 0, last, 0],
                   "values": ["Complaint Themes", 4, col, last, col], "fill": {"color": color}})
ch.set_title({"name": "Complaint themes: on-time vs late orders"}); ch.set_y_axis({"reverse": True})
ch.set_x_axis({"num_format": "0%"}); ch.set_size({"width": 620, "height": 440})
ws.insert_chart(f"A{last + 4}", ch)

ws.write("F4", "Example reviews (translated into English)", hdr)
ex = ex[ex.review_english.fillna("").str.strip() != ""].groupby("theme").head(2).reset_index(drop=True)
for i, r in ex.iterrows():
    ws.write(4 + i, 4, r["theme"], cell)
    ws.write(4 + i, 5, str(r.review_english)[:300], cell); ws.set_row(4 + i, 48)
ws.write("E4", "Theme", hdr); ws.set_column("E:E", 26)

# ================= 8. EARLY WARNING AND ACTIONS =================
ws = wb.add_worksheet("Early Warning and Actions"); setup(ws, "#C00000")
ws.set_column("A:A", 28); ws.set_column("B:D", 22); ws.set_column("E:E", 40)
ws.write("A1", "Early warning and recommended actions", title)
ws.merge_range("A2:E2", "A simple model rates each order for risk BEFORE it is delivered, using price, distance, promised "
               "time, seller history and seller handoff time. It ranks orders; it does not predict exactly.", note)
ws.set_row(1, 40)
ws.write("A4", "If support contacts only the riskiest orders...", bold)
for c, n in enumerate(["Orders flagged", "Bad reviews caught", "Times better than random"]):
    ws.write(4, c, n, hdr)
for i, (a, b, c_) in enumerate([("Riskiest 5%", 0.15, 3.0), ("Riskiest 10%", 0.24, 2.4), ("Riskiest 20%", 0.39, 1.9)]):
    ws.write(5 + i, 0, a, cell); ws.write(5 + i, 1, b, pct); ws.write(5 + i, 2, c_, dec)

ws.write("A10", "Check: does the risk ranking match reality?", bold)
for c, n in enumerate(["Risk group", "Predicted score", "Actual score", "Actual bad-review rate"]):
    ws.write(10, c, n, hdr)
for i, (g, p, a, b) in enumerate([("Riskiest 20%", 3.709, 3.908, 0.193), ("Group 2", 4.047, 4.256, 0.104),
                                  ("Group 3", 4.162, 4.379, 0.076), ("Group 4", 4.262, 4.433, 0.067),
                                  ("Safest 20%", 4.391, 4.483, 0.061)]):
    ws.write(11 + i, 0, g, cell); ws.write(11 + i, 1, p, dec); ws.write(11 + i, 2, a, dec); ws.write(11 + i, 3, b, pct)
ws.conditional_format(11, 3, 15, 3, SCALE)

ws.write("A19", "THE 3 RECOMMENDED ACTIONS", bold)
for c, n in enumerate(["Action", "Evidence", "Likely value", "What we are unsure about"]):
    ws.write(19, c, n, hdr)
acts = [
 ("1. Press the logistics partner on late deliveries, starting with the worst states",
  f"Late orders get bad reviews {p_late/p_on:.1f}x as often. Worst states: {worst_states}.",
  f"Halving late orders could cut bad reviews by about {0.5*excess_share:.0%}.",
  "Assumes lateness causes the bad review, which this data cannot prove."),
 ("2. Set a maximum handoff time for sellers and check multi-item orders",
  "Slow handoff to the carrier and larger orders were the biggest drivers of lower predicted ratings. "
  "Customers also report partial orders.",
  "Not measured yet. Run a small pilot with the slowest sellers.",
  "Partial-order cause (separate shipments) is a likely explanation, not tested."),
 ("3. Contact the riskiest 10% of orders before they go wrong",
  "The model catches 24% of bad reviews in the riskiest 10% of orders (2.4x better than random).",
  "Proactive updates or fixes on these orders.",
  "The model is modest. Test it on a small group before wide use."),
]
for i, row_ in enumerate(acts):
    for c, v in enumerate(row_): ws.write(20 + i, c, v, cell)
    ws.set_row(20 + i, 92)

wb.get_worksheet_by_name("Read Me First").activate()
wb.close()
print("Saved: excel/Olist_CX_Report.xlsx")

Saved: excel/Olist_CX_Report.xlsx


# MEMO

**To:** Head of Analytics Partnerships, Olist
**From:** [Your name]
**Re:** What is driving bad reviews, and what to do about it
**Data:** 95,560 delivered orders with a review, Jan 2017 to Aug 2018

## The answer

Late delivery is the strongest single driver of bad reviews, but not the whole story. Late orders get a 1-2 star review 6.7x as often as on-time orders (62% vs 9%), yet they are only about 7% of orders. Roughly two-thirds of bad reviews come from orders that arrived on time, mostly about wrong or missing items, quality and returns. Ops is partly right. Fixing delivery alone would cut bad reviews by about a quarter, not remove most of them.

## Three actions

**1. Use the logistics renegotiation to target Rio de Janeiro first, then the north-east.**
- The late *rate* is worst in small north-east states (AL 21%, MA 17%, SE 15%). But Rio has by far the most extra late orders: 1,456 late orders at a 12.0% late rate versus 4.4% in São Paulo.
- If Rio matched São Paulo's rate, about [RJ extra late, **check**] late orders would go away.
- *Likely value:* each late order removed avoids about 0.53 bad reviews on average (62% vs 9%), so closing Rio's gap would cut bad reviews by roughly [4%, **check**]. Halving late orders everywhere would cut them by about 14%.
- Late orders are concentrated: the 10% of sellers with the most late orders account for 74% of all late orders.

**2. Set a maximum "handoff to carrier" time for sellers and review multi-item orders.**
- In the prediction model, slow seller handoff and larger orders were the two biggest drivers of lower predicted ratings.
- Customers' comments describe partial orders ("bought 3, received 2") and having to collect parcels from the post office despite paying for home delivery.
- *Likely value:* not quantified. Run a 4-6 week pilot with the slowest sellers and compare their bad-review rate.

**3. Contact the riskiest orders before delivery.**
- A simple model ranks orders by risk before they arrive. Flagging the riskiest 10% reaches 24% of future bad reviews (2.4x better than picking at random). The riskiest 20% of orders have a 19% bad-review rate versus 6% for the safest 20%.
- *Likely value:* depends on how many flagged orders support can fix. Test on one region first.

## Evidence in one line each
- The gap between late and on-time is 53 points (95% range: 52 to 54).
- Reviews get worse step by step: 32% bad when 1-3 days late, 68% at 4-7 days, 79% at 8+ days.
- Lateness still matters after accounting for price, freight cost and distance.
- Among on-time orders, complaints are spread across quality, missing items, wrong product and returns, each about 7-14%.

## What I'm unsure about
- **Cause:** This is observational data. The link is strong and consistent, but I cannot prove delay *causes* the bad review.
- **The 28% estimate** assumes it does. The true effect could be smaller.
- **Text analysis:** Only 75% of bad reviews have a comment, and my keyword rules were about 72% accurate in a hand check, so theme shares are approximate.
- **The model is modest.** It ranks risk but cannot predict exact scores.
- **Rio vs São Paulo:** São Paulo has shorter distances, so matching its rate may not be fully achievable.
- **Partial-order cause** (separate shipments) is a likely explanation I haven't tested.